# nn-module-subclass composite — cx24: custom BatchNorm affine: gamma and beta as nn.Parameter inside a subclass

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `nn-module-subclass`, `batchnorm-affine-params`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "nn-module-subclass"
DD_ATOM_IDS = ["nn-module-subclass", "batchnorm-affine-params"]
DD_SUBTOPICS = ["PyTorch: nn.Module subclassing", "CNN: BatchNorm affine params"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

BatchNorm has TWO halves: the normalization (mean/var) and the affine (gamma, beta). This drill exercises only the AFFINE half — pretend the input is already zero-mean unit-variance and just learn the per-channel scale and shift.
- **nn-module-subclass** — `class Foo(nn.Module)` + `super().__init__()` + forward.
- **batchnorm-affine-params** — the two learnables: `gamma` (scale) and `beta` (shift), each shape `(num_features,)`. Standard init: `gamma = ones`, `beta = zeros` (so the affine is initially identity).

**Why init gamma=1, beta=0?** At init the affine `gamma * x + beta` must be the identity, so a freshly-constructed BatchNorm doesn't perturb the input distribution. Any other init would inject noise from step zero.

**Shape and broadcast.** `x` is `(N, C, H, W)`. `gamma` is `(C,)`. To broadcast across `N, H, W`, reshape `gamma` to `(1, C, 1, 1)` — same for `beta`. The output is `gamma_b * x + beta_b` where `gamma_b, beta_b` are the broadcast-reshaped versions.

**Anatomy.**
1. `super().__init__()` + store `num_features`.
2. `self.weight = nn.Parameter(t.ones(num_features))`  (gamma — convention name in PyTorch).
3. `self.bias = nn.Parameter(t.zeros(num_features))`   (beta — convention name in PyTorch).
4. `forward(x)`: reshape `weight` and `bias` to `(1, C, 1, 1)`, then `weight_b * x + bias_b`.

### Composite Exercise — custom BatchNorm affine: gamma and beta as nn.Parameter inside a subclass

**Atoms exercised together**: `nn-module-subclass`, `batchnorm-affine-params`

Define a class `MyBNAffine(nn.Module)` and a builder `cx24_build_bn_affine(num_features)`.

`MyBNAffine.__init__(self, num_features)` must:
1. `super().__init__()`.
2. Store `self.num_features = num_features`.
3. `self.weight = nn.Parameter(t.ones(num_features))`  — gamma, initialized to ALL ONES.
4. `self.bias = nn.Parameter(t.zeros(num_features))`   — beta, initialized to ALL ZEROS.

(We use the PyTorch-canonical attribute names `weight` and `bias` — the same names `nn.BatchNorm2d` uses for its affine params.)

`MyBNAffine.forward(self, x)` takes a `(N, C, H, W)` tensor and applies `weight_b * x + bias_b`, where `weight_b` and `bias_b` are the affine params reshaped to `(1, C, 1, 1)` so they broadcast across batch and spatial.

This drill skips the mean/var normalization — pretend the input is already normalized. The atom under test is the AFFINE half of BatchNorm specifically.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

class MyBNAffine(nn.Module):
    def __init__(self, num_features: int):
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


def cx24_build_bn_affine(num_features: int) -> 'MyBNAffine':
    raise NotImplementedError

def _test_cx24():
    # Case A: both params exist, right shape, right init.
    m = cx24_build_bn_affine(num_features=8)
    assert isinstance(m, nn.Module)
    assert isinstance(m.weight, nn.Parameter), 'gamma must be nn.Parameter (attr name: weight)'
    assert isinstance(m.bias, nn.Parameter), 'beta must be nn.Parameter (attr name: bias)'
    assert tuple(m.weight.shape) == (8,), f'gamma shape (C,): got {tuple(m.weight.shape)}'
    assert tuple(m.bias.shape) == (8,), f'beta shape (C,): got {tuple(m.bias.shape)}'
    assert t.equal(m.weight.data, t.ones(8)), 'gamma must init to ones — got something else'
    assert t.equal(m.bias.data, t.zeros(8)), 'beta must init to zeros — got something else'
    assert m.weight.requires_grad and m.bias.requires_grad

    # Case B: identity at init — gamma=1, beta=0 means the affine is identity.
    t.manual_seed(0)
    x = t.randn(2, 8, 4, 4)
    y = m(x)
    assert tuple(y.shape) == (2, 8, 4, 4), f'shape preservation: got {tuple(y.shape)}'
    assert t.allclose(y, x, atol=1e-6), 'init affine must be identity (gamma=1, beta=0)'

    # Case C: set gamma and beta to known per-channel values and verify the broadcast.
    m = cx24_build_bn_affine(num_features=3)
    with t.no_grad():
        m.weight.copy_(t.tensor([2.0, 0.0, -1.0]))
        m.bias.copy_(t.tensor([10.0, 20.0, 30.0]))
    x = t.ones(1, 3, 2, 2)  # all-ones input, easy to check.
    y = m(x)
    # Channel 0: 2*1 + 10 = 12.  Channel 1: 0*1 + 20 = 20.  Channel 2: -1*1 + 30 = 29.
    expected = t.tensor([12.0, 20.0, 29.0]).reshape(1, 3, 1, 1).expand(1, 3, 2, 2)
    assert t.allclose(y, expected), f'per-channel affine broken; got {y[0, :, 0, 0]}'

    # Case D: per-channel agreement vs reference manual broadcast on a random input.
    t.manual_seed(0)
    m = cx24_build_bn_affine(num_features=5)
    with t.no_grad():
        m.weight.copy_(t.randn(5))
        m.bias.copy_(t.randn(5))
    x = t.randn(3, 5, 4, 6)
    y = m(x)
    expected = m.weight.view(1, 5, 1, 1) * x + m.bias.view(1, 5, 1, 1)
    assert t.allclose(y, expected, atol=1e-6), 'affine must broadcast as gamma_b * x + beta_b'

    # Case E: matches nn.BatchNorm2d's AFFINE-only behavior (with running stats disabled and
    # eval-mode so the normalization half is a no-op identity).
    m = cx24_build_bn_affine(num_features=4)
    with t.no_grad():
        m.weight.copy_(t.tensor([1.5, 2.0, 0.5, 1.0]))
        m.bias.copy_(t.tensor([0.1, 0.2, 0.3, 0.4]))
    ref = nn.BatchNorm2d(4, affine=True, track_running_stats=False)
    with t.no_grad():
        ref.weight.copy_(m.weight)
        ref.bias.copy_(m.bias)
    # Construct an input that is already normalized per-channel so the BN normalization is a
    # no-op — then the only difference between ref and m is whether the affine matches.
    # nn.BatchNorm2d in train mode uses BATCH stats; if x is mean-0 var-1 per channel within
    # the (N, H, W) slice, the normalization yields x back (modulo eps).
    N, C, H, W = 8, 4, 3, 3
    raw = t.randn(N, C, H, W)
    # Standardize each channel across the (N, H, W) axes so BN's normalization is a no-op.
    axes = (0, 2, 3)
    mean = raw.mean(dim=axes, keepdim=True)
    std = raw.std(dim=axes, keepdim=True, unbiased=False)
    x_std = (raw - mean) / (std + 1e-8)
    y_ours = m(x_std)
    y_ref = ref(x_std)
    assert t.allclose(y_ours, y_ref, atol=1e-3), 'must match nn.BatchNorm2d affine on already-normalized input'

    # Case F: .parameters() returns exactly 2 tensors.
    m = cx24_build_bn_affine(num_features=6)
    ps = list(m.parameters())
    assert len(ps) == 2, f'expected 2 params (weight, bias); got {len(ps)}'
    named = dict(m.named_parameters())
    assert set(named.keys()) == {'weight', 'bias'}
    _dd_passed.add('cx24')

_test_cx24()

<details><summary>Show solution — cx24</summary>

```python
class MyBNAffine(nn.Module):
    def __init__(self, num_features: int):
        # Atom A (nn-module-subclass): scaffold first.
        super().__init__()
        self.num_features = num_features
        # Atom B (batchnorm-affine-params): gamma (scale) inits to ONES so the affine is
        # identity at construction. PyTorch attribute name is `weight` for consistency
        # with nn.BatchNorm2d.
        self.weight = nn.Parameter(t.ones(num_features))
        # beta (shift) inits to ZEROS — also for identity init. Attribute name is `bias`.
        self.bias = nn.Parameter(t.zeros(num_features))

    def forward(self, x):
        # x shape: (N, C, H, W). gamma / beta are (C,) — reshape to (1, C, 1, 1) so they
        # broadcast across batch and spatial axes.
        gamma_b = self.weight.view(1, -1, 1, 1)
        beta_b = self.bias.view(1, -1, 1, 1)
        return gamma_b * x + beta_b


def cx24_build_bn_affine(num_features: int) -> 'MyBNAffine':
    return MyBNAffine(num_features)
```

Init choice is non-negotiable: gamma=1, beta=0 makes the affine the identity at construction, so a freshly-built BatchNorm doesn't shift the input distribution. The broadcast reshape `(C,) -> (1, C, 1, 1)` is the canonical pattern for any per-channel affine on a 4-D feature map.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx24'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx24',
        'subtopics': ["PyTorch: nn.Module subclassing", "CNN: BatchNorm affine params"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()